In [ ]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import spacy

In [ ]:
nlp = spacy.load("de_core_news_md")
BASE_SEARCH = "https://www.bz-berlin.de/suche"
CLIMATE_KEYWORDS = ["klima", "klimawandel", "klimakrise", "erderwärmung", "globale erwärmung", "treibhauseffekt", "treibhausgas",
    "co2", "kohlendioxid", "emission", "emissionen", "klimaschutz", "klimapolitik", "klimaneutral", "dekarbonisierung",
    
    "klimaziel", "klimaziele", "co2-preis", "emissionshandel", "klimaschutzgesetz", "paris-abkommen", "pariser abkommen",
    "eu-klimapaket", "green deal", "klimaplan", "klimabericht", "ipcc",

    "energiewende", "erneuerbare energien", "windkraft", "solarenergie", "photovoltaik", "wasserstoff",
    "kohlekraft", "kohleausstieg", "atomkraft", "stromnetz", "netzausbau", "e-mobilität", "verbrennerverbot", "industrieemissionen",
    
    "hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter",
    "gletscherschmelze", "meeresspiegel", "wasserknappheit","klimafolgen", "artensterben",

    "klimarisiko", "nachhaltigkeit", "nachhaltige investitionen", "esg", "grüne finanzierung", "klimafonds", "energiepreise", "strompreis", "gaspreis",

    "landwirtschaft", "trockenheit", "ernteschäden", "umweltschutz", "biodiversität", "naturschutz", "ökosystem", "umweltpolitik",

    "fridays for future", "klimaaktivisten","klimaprotest", "letzte generation", "klimabewegung","klimadebatte"]
STRICT_KEYWORDS = ["klima", "klimakrise", "klimawandel", "erderwärmung", "co2", "kohlendioxid", "emission", "emissionen",
    "energiewende", "klimaschutz", "klimaneutral", "treibhauseffekt", "treibhausgas",
    "hitzewelle", "dürre", "hochwasser", "wasserknappheit", "starkregen", "waldbrand"]
YEARS = range(2020,2026)
OUTPUT_FILE = "bz_scrapped.csv"
INVALID_URL_PATTERNS = ["/video/","/bilder/","/spiele/", "/angebote/","/shopping/"]
CLASSIFICATION_KEYWORDS = {
    "Climate_Policy": ["gesetz", "politik", "regierung", "beschluss", "verordnung", "ziel", "klimaziel", "bundestag", "eu", "parlament", "ministerium"],
    "Climate_Science": ["studie", "forschung", "wissenschaft", "ipcc", "daten", "analyse", "bericht", "modell", "forscher"],
    "Energy_Transition": ["energiewende", "erneuerbar", "solar", "windkraft","kohlekraft", "atomkraft", "wasserstoff", "stromnetz"],
    "Climate_Economy": ["kosten", "industrie", "wirtschaft", "markt", "unternehmen", "investition", "preis", "arbeitsplätze"],
    "Climate_Activism": ["protest", "demonstration", "aktivisten","fridays for future", "ngo", "bewegung"],
    "Climate_Impact": ["hitzewelle", "dürre", "hochwasser", "überschwemmung","starkregen", "waldbrand", "extremwetter"],
    "Climate_Geopolitics": ["china", "usa", "eu", "russland", "international", "global", "weltweit", "g7", "g20"],
    "Climate_Opinion": ["meinung", "kommentar", "kolumne", "leitartikel", "debatte", "gastbeitrag"]}

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
chrome_options.add_argument(r"--user-data-dir=C:/selenium_chrome_profile")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),options=chrome_options)
wait = WebDriverWait(driver,20)

In [ ]:
def human_sleep(a=2,b=5):
    time.sleep(random.uniform(a,b))
def safe_get(url):
    try:
        driver.get(url)
        human_sleep(2,4)
        return True
    except:
        return False
def clean_text(text):
    if not text:
        return None
    return re.sub(r"\s+"," ",text).strip()
def extract_author():
    try:
        author = driver.find_element(By.CSS_SELECTOR,"a[rel='author']")
        return author.text.strip()
    except:
        pass
    try:
        container = driver.find_element(By.CSS_SELECTOR,'div[data-type="author"]')
        span = container.find_element(By.TAG_NAME,"span")
        return span.text.strip()
    except:
        pass
    return None
def extract_date():
    try:
        return driver.find_element(
        By.CSS_SELECTOR,"div.entry-meta small").text.strip()
    except:
        return None
def extract_article():
    paragraphs = driver.find_elements(By.TAG_NAME,"p")
    clean = []
    for p in paragraphs:
        t = p.text.strip()
        if len(t) < 40:
            continue
        if any(x in t.lower() for x in [ "anzeige","newsletter", "abonnieren","quelle:", "bild:"]):
            continue
        clean.append(t)
    if len(clean) < 3:
        return None,None
    intro = clean[0]
    content = clean_text(" ".join(clean))
    return intro,content
def strict_relevance(title,content):
    title = title.lower()
    content_l = content.lower()
    lead = " ".join(content_l.split()[:300])
    title_hit = any(k in title for k in STRICT_KEYWORDS)
    lead_hits = sum(k in lead for k in STRICT_KEYWORDS)
    return title_hit or lead_hits >= 3
def classify_article(title,intro,content):
    text = f"{title} {intro} {' '.join(content.split()[:300])}".lower()
    scores = {}
    for cat,keywords in CLASSIFICATION_KEYWORDS.items():
        scores[cat] = sum(k in text for k in keywords)
    best = max(scores,key=scores.get)
    return best if scores[best] > 0 else "Other_Climate"
def analyze_text(text):
    doc = nlp(text[:4000])
    actors = sorted(set(ent.text for ent in doc.ents if ent.label_ in ["PER","ORG"]))
    sentences = list(doc.sents)
    return actors,len(actors),len(sentences),len(text)

In [ ]:
rows = []
visited = set()

In [ ]:
for keyword in CLIMATE_KEYWORDS:
    print(f"\nKeyword: {keyword}")
    for page in range(1,10):
        print(f"Page {page}")
        search_url = f"{BASE_SEARCH}?q={keyword}&page={page}"
        if not safe_get(search_url):
            continue
        wait.until(EC.presence_of_element_located((By.TAG_NAME,"article")))
        articles = driver.find_elements(By.CSS_SELECTOR,"article a[href]")
        if len(articles) == 0:
            break
        urls = []
        for a in articles:
            url = a.get_attribute("href")
            if not url:
                continue
            if not url.startswith("https://www.bz-berlin.de"):
                continue
            if any(bad in url for bad in INVALID_URL_PATTERNS):
                continue
            if url in visited:
                continue
            visited.add(url)
            urls.append(url)
        for url in urls:
            if not safe_get(url):
                continue
            try:
                wait.until(
                EC.presence_of_element_located((By.TAG_NAME,"h1")))
                headline = driver.find_element(By.TAG_NAME,"h1").text.strip()
                intro,content = extract_article()
                if not content:
                    continue
                date = extract_date()
                if not date:
                    continue
                if not any(str(y) in date for y in YEARS):
                    continue
                if not strict_relevance(headline,content):
                    continue
                actors,actor_count,sent_count,length = analyze_text(content)
                article_class = classify_article(headline,intro,content)
                rows.append({
                    "URL": url,
                    "Source": "B.Z.",
                    "Language": "de",
                    "Published_Date": date,
                    "Keyword_Matched": keyword,
                    "Article_Classification": article_class,
                    "Headline": headline,
                    "Intro": intro,
                    "Content": content,
                    "Content_Length": length,
                    "Sentence_Count": sent_count,
                    "Actors": ", ".join(actors),
                    "Actor_Count": actor_count,
                    "Author": extract_author()
                })
                print("Saved:")
                human_sleep()
            except:
                continue

In [ ]:
df = pd.DataFrame(rows)
df.drop_duplicates(subset=["URL"],inplace=True)
df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")
driver.quit()
print(f"File: {OUTPUT_FILE}")
print(f"Total strict climate articles: {len(df)}")